# Data Cleaning

**Input:** `data/raw/customer_support/customer_support_tickets.csv` — 8,469 rows  
**Output:** `data/processed/cleaned_tasks.csv`

## Important finding about `{product_purchased}`
ALL 8,469 rows contain the substring `{product_purchased}` — the entire dataset was generated
from templates. Filtering on this pattern removes everything. Strategy: use
`drop_duplicates(subset=['Ticket Description'])` to remove the 392 repeated/identical template rows instead.

## Cleaning steps
| Step | Action | Rows lost |
|---|---|---|
| Drop 3 high-null columns | `Customer Satisfaction Rating`, `Time to Resolution`, `First Response Time` | 0 |
| Fill `Resolution` nulls | Fill with `Unknown` | 0 |
| Remove exact duplicate rows | Full-row duplicates | small |
| Remove duplicate descriptions | 392 repeated identical descriptions | ~392 |
| Standardize labels | lowercase + strip whitespace on targets | 0 |
| Clean text | lowercase, strip email/URL/digits/punctuation | 0 |
| Drop very short cleaned text | < 5 words after cleaning | small |

In [1]:
# DIAGNOSTIC CELL — run this alone first to verify filter behaviour before cleaning
import pandas as pd, re

df_test = pd.read_csv('../data/raw/customer_support/customer_support_tickets.csv')
print(f'Raw shape: {df_test.shape}')                                     # expect (8469, 17)

mask = df_test['Ticket Description'].str.contains(
    r'\{product_purchased\}', regex=True, na=False
)
print(f'Rows WITH placeholder   : {mask.sum()}')                         # if ~8469 -> all rows templated
print(f'Rows WITHOUT placeholder: {(~mask).sum()}')                      # if ~0   -> cannot use this filter

dup_desc = df_test.duplicated(subset=['Ticket Description'], keep=False)
print(f'Duplicate descriptions  : {dup_desc.sum()}')                     # expect ~392

del df_test
print('\nDiagnostic complete. Proceed with cleaning cells below.')

Raw shape: (8469, 17)
Rows WITH placeholder   : 8469
Rows WITHOUT placeholder: 0
Duplicate descriptions  : 439

Diagnostic complete. Proceed with cleaning cells below.


In [2]:
import pandas as pd
import numpy as np
import re
import os

RAW_PATH     = '../data/raw/customer_support/customer_support_tickets.csv'
CLEANED_PATH = '../data/processed/cleaned_tasks.csv'
os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f'Shape after loading: {df.shape}')   # expect (8469, 17)
print(df.columns.tolist())

Shape after loading: (8469, 17)
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']


In [3]:
cols_to_drop = [
    'Customer Satisfaction Rating',
    'Time to Resolution',
    'First Response Time'
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]  # safety check
df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped: {cols_to_drop}')
print(f'Shape after dropping unused columns: {df.shape}')   # expect (8469, 14)
print(df.columns.tolist())

Dropped: ['Customer Satisfaction Rating', 'Time to Resolution', 'First Response Time']
Shape after dropping unused columns: (8469, 14)
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel']


In [4]:
df['Resolution'] = df['Resolution'].fillna('Unknown')

print(f'Shape after filling nulls: {df.shape}')  # still (8469, 14)
print(df.isnull().sum())

Shape after filling nulls: (8469, 14)
Ticket ID             0
Customer Name         0
Customer Email        0
Customer Age          0
Customer Gender       0
Product Purchased     0
Date of Purchase      0
Ticket Type           0
Ticket Subject        0
Ticket Description    0
Ticket Status         0
Resolution            0
Ticket Priority       0
Ticket Channel        0
dtype: int64


In [5]:
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Exact duplicate rows removed: {before - len(df)}')
print(f'Shape after removing exact duplicates: {df.shape}')

Exact duplicate rows removed: 0
Shape after removing exact duplicates: (8469, 14)


In [6]:
# NOTE: {product_purchased} appears in ALL rows — entire dataset is template-generated.
# Filtering on that substring removes everything. Instead remove duplicate descriptions
# (the 392 repeated identical template strings) using drop_duplicates.

before = len(df)

# Verify before removing
dup_count = df.duplicated(subset=['Ticket Description'], keep='first').sum()
print(f'Duplicate descriptions to remove: {dup_count}')  # expect ~392

df.drop_duplicates(subset=['Ticket Description'], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Rows removed: {before - len(df)}')
print(f'Shape after removing duplicate descriptions: {df.shape}')  # expect ~8077

Duplicate descriptions to remove: 392
Rows removed: 392
Shape after removing duplicate descriptions: (8077, 14)


In [7]:
print('Ticket Type before:', df['Ticket Type'].unique())
print('Ticket Priority before:', df['Ticket Priority'].unique())

df['Ticket Type']     = df['Ticket Type'].str.lower().str.strip()
df['Ticket Priority'] = df['Ticket Priority'].str.lower().str.strip()

print('\nTicket Type after:')
print(df['Ticket Type'].value_counts())
print('\nTicket Priority after:')
print(df['Ticket Priority'].value_counts())
print(f'\nShape after standardizing labels: {df.shape}')

Ticket Type before: <ArrowStringArray>
[     'Technical issue',      'Billing inquiry', 'Cancellation request',
      'Product inquiry',       'Refund request']
Length: 5, dtype: str
Ticket Priority before: <ArrowStringArray>
['Critical', 'Low', 'High', 'Medium']
Length: 4, dtype: str

Ticket Type after:
Ticket Type
refund request          1659
technical issue         1651
cancellation request    1619
product inquiry         1577
billing inquiry         1571
Name: count, dtype: int64

Ticket Priority after:
Ticket Priority
medium      2090
critical    2030
high        2001
low         1956
Name: count, dtype: int64

Shape after standardizing labels: (8077, 14)


In [8]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    
    # Remove leftover placeholder artifacts from template rows
    text = re.sub(r'\{?product_?purchased\}?', '', text)  # catches {product_purchased} and productpurchased
    
    text = re.sub(r'\S+@\S+', '', text)           # remove emails
    text = re.sub(r'http\S+|www\S+', '', text)     # remove URLs
    text = re.sub(r'\d+', '', text)                # remove digits
    text = re.sub(r'[^a-z\s]', '', text)           # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return text

In [9]:
df['full_text']  = df['Ticket Subject'].astype(str) + ' ' + df['Ticket Description'].astype(str)
df['clean_text'] = df['full_text'].apply(clean_text)

print(f'Shape after adding text columns: {df.shape}')
print('\nSample clean_text:')
print(df['clean_text'].iloc[0])

Shape after adding text columns: (8077, 16)

Sample clean_text:
product setup im having an issue with the please assist your billing zip code is we appreciate that you have requested a website address please double check your email address ive tried troubleshooting steps mentioned in the user manual but the issue persists


In [10]:
df['word_count_temp'] = df['clean_text'].apply(lambda x: len(x.split()))

print('Word count distribution after cleaning:')
print(df['word_count_temp'].describe())
print(f'\nRows with fewer than 5 words: {(df["word_count_temp"] < 5).sum()}')

df = df[df['word_count_temp'] >= 5]
df.drop(columns=['word_count_temp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape after dropping very short text: {df.shape}')

Word count distribution after cleaning:
count    8077.000000
mean       46.914696
std         7.873318
min        22.000000
25%        43.000000
50%        49.000000
75%        53.000000
max        62.000000
Name: word_count_temp, dtype: float64

Rows with fewer than 5 words: 0
Shape after dropping very short text: (8077, 16)


In [11]:
print('=== FINAL SANITY CHECK ===')
print(f'Final shape: {df.shape}')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nTicket Type distribution:\n{df["Ticket Type"].value_counts()}')
print(f'\nTicket Priority distribution:\n{df["Ticket Priority"].value_counts()}')
print(f'\nDuplicate descriptions remaining: {df.duplicated(subset=["Ticket Description"]).sum()}')  # expect 0
print(f'\nSample clean_text:')
for t in df['clean_text'].sample(3, random_state=1).tolist():
    print(' -', t[:100])

=== FINAL SANITY CHECK ===
Final shape: (8077, 16)

Null counts:
Ticket ID             0
Customer Name         0
Customer Email        0
Customer Age          0
Customer Gender       0
Product Purchased     0
Date of Purchase      0
Ticket Type           0
Ticket Subject        0
Ticket Description    0
Ticket Status         0
Resolution            0
Ticket Priority       0
Ticket Channel        0
full_text             0
clean_text            0
dtype: int64

Ticket Type distribution:
Ticket Type
refund request          1659
technical issue         1651
cancellation request    1619
product inquiry         1577
billing inquiry         1571
Name: count, dtype: int64

Ticket Priority distribution:
Ticket Priority
medium      2090
critical    2030
high        2001
low         1956
Name: count, dtype: int64

Duplicate descriptions remaining: 0

Sample clean_text:
 - account access im having an issue with the please assist i have a new account in the system and a ne
 - hardware issue ive noti

In [12]:
df.to_csv(CLEANED_PATH, index=False)

size_mb = os.path.getsize(CLEANED_PATH) / (1024 * 1024)
print(f'Saved  : {CLEANED_PATH}')
print(f'Size   : {size_mb:.2f} MB')
print(f'Shape  : {df.shape}')
print('\nReady for 04_nlp_preprocessing.ipynb')

Saved  : ../data/processed/cleaned_tasks.csv
Size   : 7.96 MB
Shape  : (8077, 16)

Ready for 04_nlp_preprocessing.ipynb
